<a href="https://colab.research.google.com/github/juhan9437-dotcom/haccp_dashboard/blob/main/07_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 양돈 사료 곰팡이독소 QA/QC Dashboard - Google Colab 원셀 실행본
# Main screen: 16:9 화면을 고려한 컴팩트 레이아웃
# Dash + Plotly + Bootstrap / 전처리·별도 데이터 파일 불필요
# ============================================================

import sys, subprocess, importlib.util, socket, threading, time, base64, calendar

# 1) 필요한 라이브러리 자동 설치
required = {
    "dash": "dash>=2.17",
    "dash_bootstrap_components": "dash-bootstrap-components>=1.6",
    "plotly": "plotly>=5.20",
    "pandas": "pandas>=2.0",
    "numpy": "numpy>=1.24",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc

# 2) 공개 모니터링 결과 + 대한민국 권고기준 목록
# 근거: 「사료 등의 기준 및 규격」 [별표 16] 2의2. 관리를 추천하는 곰팡이독소
# 권고 수준은 법적 허용기준과 구분합니다. 이번 공개사례에서 개별 측정값이 확인된 권고독소는 DON과 ZEN입니다.
N_PUBLIC_SAMPLES = 17
RULE_URL = "https://www.law.go.kr/admRulLsInfoP.do?admRulSeq=2100000238528"
CASE_URL = "https://koreapork.or.kr/board/view1?Ncode=pignews&gubun=&gubuncode=&keyfield=&keyword=&number=51583&page=1"

# 2026년 공개 모니터링에서 분석한 네 가지 독소의 결과
PUBLIC_TOXIN_SUMMARY = pd.DataFrame([
    {"독소": "아플라톡신", "검출건수": 0, "초과건수": 0, "검사건수": 17, "분류": "법정 관리항목"},
    {"독소": "오크라톡신A", "검출건수": 0, "초과건수": 0, "검사건수": 17, "분류": "법정 관리항목"},
    {"독소": "보미톡신(DON)", "검출건수": 17, "초과건수": 1, "검사건수": 17, "분류": "관리 권고항목"},
    {"독소": "제랄레논(ZEN)", "검출건수": 2, "초과건수": 1, "검사건수": 17, "분류": "관리 권고항목"},
])

# 현행 대한민국 「사료 등의 기준 및 규격」 별표 16의 권고대상 독소 4종
RECOMMENDED_TOXINS = {
    "DON": {
        "label": "보미톡신(DON)",
        "option": "보미톡신(DON) · 돼지용 배합사료 900 ppb",
        "standard": "돼지용 배합사료 900 ppb",
        "public_note": "이번 공개 모니터링에서 DON은 17건 모두 검출되었고, 1건이 보도자료에 제시된 권고기준 900 ppb를 초과했습니다.",
    },
    "ZEN": {
        "label": "제랄레논(ZEN)",
        "option": "제랄레논(ZEN) · 돼지 사료 100 / 250 ppb*",
        "standard": "포유자돈·이유돈·번식용모돈·임신돈·포유돈 등 100 ppb; 기타 육성·비육돈용 250 ppb",
        "public_note": "보도자료는 100 ppb를 기준으로 초과 사례를 제시했습니다. G·I 농장의 개별 급여단계가 공개되지 않아, 차트는 보도자료의 판정 기준 대비로 표시하며 별표 16에 따른 재판정은 하지 않습니다.",
    },
    "FUM": {
        "label": "푸모니신(B1+B2)",
        "option": "푸모니신(B1+B2) · 돼지용 배합사료 5,000 ppb",
        "standard": "돼지용 배합사료 5,000 ppb",
        "public_note": "이번 공개 모니터링의 분석 대상에 푸모니신은 포함되지 않아 공개 측정값이 없습니다.",
    },
    "T2HT2": {
        "label": "T-2/HT-2 독소",
        "option": "T-2/HT-2 독소 · 배합사료 250 ppb",
        "standard": "전체 배합사료 250 ppb",
        "public_note": "이번 공개 모니터링의 분석 대상에 T-2/HT-2는 포함되지 않아 공개 측정값이 없습니다.",
    },
}

# 공개 기사에서 개별 농도까지 확인된 수치만 입력합니다.
# ZEN의 100 ppb는 기사에 제시된 비교 기준이며, 별표 16에서는 급여단계별 100/250 ppb로 구분됩니다.
PUBLIC_KNOWN_RESULTS = pd.DataFrame([
    {"독소코드": "DON", "독소": "보미톡신(DON)", "농장": "F업체 · 농장 10", "농도_ppb": 1088.3, "기사기준_ppb": 900.0, "결과": "권고기준 초과"},
    {"독소코드": "DON", "독소": "보미톡신(DON)", "농장": "G업체 · 농장 12", "농도_ppb": 547.0, "기사기준_ppb": 900.0, "결과": "권고기준 이내"},
    {"독소코드": "DON", "독소": "보미톡신(DON)", "농장": "I업체 · 농가", "농도_ppb": 332.2, "기사기준_ppb": 900.0, "결과": "권고기준 이내"},
    {"독소코드": "ZEN", "독소": "제랄레논(ZEN)", "농장": "G업체 · 농장 12", "농도_ppb": 134.2, "기사기준_ppb": 100.0, "결과": "권고기준 초과(보도자료 판정)"},
    {"독소코드": "ZEN", "독소": "제랄레논(ZEN)", "농장": "I업체 · 농가", "농도_ppb": 91.29, "기사기준_ppb": 100.0, "결과": "권고기준 이내(보도자료 판정)"},
])
PUBLIC_KNOWN_RESULTS["기준대비_pct"] = PUBLIC_KNOWN_RESULTS["농도_ppb"] / PUBLIC_KNOWN_RESULTS["기사기준_ppb"] * 100
PUBLIC_KNOWN_RESULTS["막대색"] = np.where(PUBLIC_KNOWN_RESULTS["기준대비_pct"] > 100, "권고기준 초과", "권고기준 이내")
# 3) 운영 KPI 계산용 템플릿 기록
# 실제 기업 운영자료를 받으면 이 부분의 OPS_DF 생성 부분을 해당 자료로 교체합니다.
rng = np.random.default_rng(20261029)
months = pd.date_range("2026-05-01", "2026-08-01", freq="MS")
suppliers = [f"공급사 {x}" for x in "ABCDEF"]
feed_types = ["육성·비육돈용", "임신돈용"]

supplier_storage_bias = {s: b for s, b in zip(suppliers, [0.000, 0.015, 0.030, 0.010, 0.045, 0.020])}
supplier_inspection_bias = {s: b for s, b in zip(suppliers, [0.000, 0.015, 0.030, 0.010, 0.020, 0.040])}
supplier_sop_bias = {s: b for s, b in zip(suppliers, [0.000, 0.010, 0.025, 0.015, 0.030, 0.020])}
month_storage_bias = {"2026-05": 0.000, "2026-06": 0.015, "2026-07": 0.050, "2026-08": 0.070}
month_inspection_bias = {"2026-05": 0.000, "2026-06": 0.005, "2026-07": 0.015, "2026-08": 0.025}

ops_rows = []
lot_counter = 1
for month in months:
    month_key = month.strftime("%Y-%m")
    days_in_month = calendar.monthrange(month.year, month.month)[1]
    for supplier in suppliers:
        for feed_type in feed_types:
            for _ in range(3):
                event_day = int(rng.integers(1, days_in_month + 1))
                event_date = pd.Timestamp(year=month.year, month=month.month, day=event_day)
                inspection_p = np.clip(0.965 - supplier_inspection_bias[supplier] - month_inspection_bias[month_key], 0.80, 0.99)
                inspection_done = bool(rng.random() < inspection_p)
                # 보관환경 점검 항목: 수분·수분활성도, 온습도, 결로·누수 (3개 영역)
                environment_checks_per_lot = 3
                environment_p = np.clip(0.045 + supplier_storage_bias[supplier] + month_storage_bias[month_key], 0.02, 0.30)
                environment_deviation_count = int(rng.binomial(environment_checks_per_lot, environment_p))
                sop_p = np.clip(0.965 - supplier_sop_bias[supplier] - month_inspection_bias[month_key] / 2, 0.80, 0.99)
                sop_pass = bool(rng.random() < sop_p)
                if not inspection_done:
                    status = "검사 미완료"
                elif environment_deviation_count > 0:
                    status = "보관환경 이탈"
                elif not sop_pass:
                    status = "SOP 확인 필요"
                else:
                    status = "정상"
                ops_rows.append({
                    "날짜": event_date, "월": month_key, "공급사": supplier, "사료종류": feed_type,
                    "로트": f"LOT-{month_key.replace('-', '')}-{lot_counter:04d}",
                    "검사완료": inspection_done, "환경점검항목수": environment_checks_per_lot,
                    "환경이탈건수": environment_deviation_count, "SOP준수": sop_pass,
                    "상태": status,
                })
                lot_counter += 1
OPS_DF = pd.DataFrame(ops_rows)

# 4) 파스텔 하늘빛·노을빛 스타일
NAVY = "#466781"
SKY = "#82CFF0"
SKY_DARK = "#5AB9E2"
CORAL = "#FF9A7A"
RED = "#FF0000"  # 권고기준을 표시하는 점선은 순수 빨강
GREEN = "#78CBB8"
AMBER = "#F5B98E"
LAVENDER = "#C9C7F5"
MUTED = "#71879B"
LIGHT_BG = "#F4FAFF"
PANEL_BORDER = "#DDECF5"
PLOT_TEMPLATE = "plotly_white"

# 외부 다운로드가 필요 없는 파스텔 톤 SVG 일러스트
svg_art = '''<svg xmlns="http://www.w3.org/2000/svg" width="240" height="118" viewBox="0 0 240 118">
<defs><linearGradient id="g" x1="0" y1="0" x2="1" y2="1"><stop offset="0" stop-color="#DDF4FF"/><stop offset="1" stop-color="#EAE8FF"/></linearGradient></defs>
<rect width="240" height="118" rx="18" fill="url(#g)"/><circle cx="205" cy="25" r="42" fill="#FFFFFF" opacity=".45"/><circle cx="27" cy="101" r="35" fill="#B7E8F8" opacity=".45"/>
<g transform="translate(18,25)" stroke="#76BFD9" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><ellipse cx="45" cy="42" rx="29" ry="23" fill="#FFFFFF"/><circle cx="69" cy="35" r="17" fill="#FFFFFF"/><path d="M58 22 L55 7 L68 21 Z" fill="#FFD8CF"/><path d="M73 21 L86 10 L84 29 Z" fill="#FFD8CF"/><ellipse cx="85" cy="40" rx="11" ry="7" fill="#FFB8A4" stroke="#FFB8A4"/><circle cx="72" cy="32" r="2.5" fill="#466781" stroke="none"/><path d="M27 60 L25 73 L37 73 L40 61 M52 61 L54 73 L66 73 L63 58" fill="#FFFFFF"/><path d="M17 38 Q5 28 13 22" fill="none"/></g>
<text x="120" y="47" font-family="Arial,sans-serif" font-size="11" font-weight="700" fill="#466781">FEED QUALITY</text><text x="120" y="64" font-family="Arial,sans-serif" font-size="9" fill="#5A9DBA">MYCOTOXIN QA/QC</text><text x="120" y="81" font-family="Arial,sans-serif" font-size="8" fill="#8492C1">QUALITY MONITORING</text></svg>'''
SVG_URI = "data:image/svg+xml;base64," + base64.b64encode(svg_art.encode("utf-8")).decode("ascii")


def polish_fig(fig, height=300, legend=True):
    fig.update_layout(
        template=PLOT_TEMPLATE, height=height,
        margin=dict(l=18, r=13, t=38, b=27),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Arial, 'Noto Sans KR', sans-serif", color="#405D72", size=11),
        title=dict(x=0.02, xanchor="left", font=dict(size=14, color=NAVY)),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, font=dict(size=10)) if legend else dict(visible=False),
        hoverlabel=dict(bgcolor="#FFFFFF", bordercolor="#CBE9F7", font_size=12),
    )
    fig.update_xaxes(showgrid=False, zeroline=False, linecolor="#DDECF5")
    fig.update_yaxes(gridcolor="#EAF3F8", zeroline=False, linecolor="#DDECF5")
    return fig

# 공개 모니터링에서 분석한 네 가지 독소의 검출·초과 건수
public_long = PUBLIC_TOXIN_SUMMARY.melt(
    id_vars=["독소", "검사건수", "분류"], value_vars=["검출건수", "초과건수"],
    var_name="구분", value_name="건수"
)
public_long["구분"] = public_long["구분"].map({"검출건수": "검출 건수", "초과건수": "기준 초과 건수"})
fig_public = px.bar(
    public_long, x="독소", y="건수", color="구분", barmode="group",
    color_discrete_map={"검출 건수": SKY, "기준 초과 건수": CORAL},
    category_orders={"독소": ["아플라톡신", "오크라톡신A", "보미톡신(DON)", "제랄레논(ZEN)"]},
    custom_data=["분류", "검사건수"],
)
fig_public.update_traces(
    marker_line_width=0,
    hovertemplate="%{x}<br>%{fullData.name}: %{y}건<br>항목 구분: %{customdata[0]}<br>검사 검체: %{customdata[1]}건<extra></extra>"
)
fig_public.update_yaxes(title="검체 건수", range=[0, 19], dtick=5)
fig_public.update_layout(barmode="group", bargap=0.27, bargroupgap=0.12, showlegend=True)
fig_public = polish_fig(fig_public, 310)


def make_ratio_fig(toxin_choice="ALL"):
    """공개된 측정값만 표시. 값이 없는 독소는 공식 권고수준과 미공개 안내를 보여줍니다."""
    choice = toxin_choice or "ALL"
    selected_codes = ["DON", "ZEN"] if choice == "ALL" else [choice]
    dff = PUBLIC_KNOWN_RESULTS[PUBLIC_KNOWN_RESULTS["독소코드"].isin(selected_codes)].copy()

    if dff.empty:
        item = RECOMMENDED_TOXINS.get(choice)
        label = item["label"] if item else "권고기준 대상 독소"
        standard = item["standard"] if item else "별표 16 참고"
        fig = go.Figure()
        fig.add_annotation(
            x=0.5, y=0.60, xref="paper", yref="paper", showarrow=False,
            text=f"<b>{label}</b><br>이번 공개 모니터링에서 개별 측정값이 확인되지 않았습니다.<br><span style='font-size:12px'>권고수준: {standard}</span>",
            font=dict(size=13, color=NAVY), align="center", bgcolor="#F7FCFF",
            bordercolor="#D9ECF7", borderwidth=1, borderpad=16,
        )
        fig.update_xaxes(visible=False, range=[0, 1])
        fig.update_yaxes(visible=False, range=[0, 1])
        fig.update_layout(title="공개 측정값", showlegend=False)
        return polish_fig(fig, 310, legend=False)

    dff = dff.sort_values("기준대비_pct", ascending=True)
    fig = px.bar(
        dff, x="기준대비_pct", y="농장", orientation="h", color="막대색",
        color_discrete_map={"권고기준 초과": CORAL, "권고기준 이내": SKY},
        custom_data=["독소", "농도_ppb", "기사기준_ppb", "결과"],
    )
    # 기준선은 순수 빨강. ZEN은 기사의 적용 기준과 법령상 단계별 권고수준을 주의문으로 구분합니다.
    fig.add_vline(x=100, line_dash="dash", line_width=2.5, line_color="#FF0000",
                  annotation_text="기준 100%", annotation_position="top", annotation_font_color="#FF0000")
    fig.update_traces(
        hovertemplate="%{y}<br>독소: %{customdata[0]}<br>공개 측정값: %{customdata[1]:,.2f} ppb<br>보도자료 적용 기준: %{customdata[2]:,.2f} ppb<br>기준 대비: %{x:.1f}%<br>%{customdata[3]}<extra></extra>",
        marker_line_width=0,
    )
    max_x = max(150, float(dff["기준대비_pct"].max()) + 12)
    fig.update_xaxes(title="보도자료 적용 권고기준 대비 (%)", range=[0, max_x])
    fig.update_yaxes(title="", automargin=True)
    return polish_fig(fig, 310, legend=False)

# 5) Dash 앱 및 16:9 메인 대시보드
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP], title="양돈 사료 곰팡이독소 QA/QC")
app.index_string = '''<!DOCTYPE html>
<html><head>{%metas%}<title>{%title%}</title>{%favicon%}{%css%}
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
:root { --navy:#466781; --sky:#82CFF0; --coral:#FF9A7A; --bg:#F4FAFF; --muted:#71879B; }
* { box-sizing:border-box; }
html, body { margin:0; padding:0; background:var(--bg); font-family:Arial,'Noto Sans KR',sans-serif; color:#405D72; }
.container-fluid { width:100%; max-width:none; padding:0!important; }
.dashboard-layout { width:100%; height:100vh; min-height:650px; max-height:100vh; overflow:hidden; display:grid; grid-template-columns:164px minmax(0,1fr); gap:10px; padding:10px 12px; }
.sidebar { min-width:0; height:100%; border:1px solid #DDECF5; border-radius:17px; padding:10px; background:linear-gradient(180deg,#E5F6FF 0%,#F0F2FF 55%,#F7FBFF 100%); box-shadow:0 4px 16px rgba(92,157,191,.08); display:flex; flex-direction:column; overflow:hidden; }
.sidebar-image { display:block; width:100%; height:auto; max-height:88px; object-fit:contain; border-radius:11px; margin-bottom:13px; }
.side-eyebrow { color:#699DB9; font-size:9px; font-weight:850; letter-spacing:1.3px; margin:0 0 7px 2px; }
.side-title { color:#365D78; font-size:clamp(17px,1.3vw,22px); line-height:1.32; letter-spacing:-.7px; font-weight:900; margin:0 0 7px; overflow-wrap:break-word; }
.side-sub { color:#6D879B; font-size:10px; line-height:1.55; margin:0 0 12px; white-space:pre-line; }
.side-divider { height:1px; background:#D4E8F3; margin:5px 0 12px; }
.side-nav { display:grid; gap:7px; }
.side-link { border:1px solid rgba(166,206,229,.55); border-radius:9px; padding:7px 8px; color:#68869B; font-size:10px; font-weight:750; background:rgba(255,255,255,.46); }
.side-link.active { background:#FFFFFF; border-color:#B8E5F7; color:#376E8D; box-shadow:0 2px 7px rgba(92,157,191,.08); }
.side-number { display:inline-block; color:#70BBD9; margin-right:5px; font-size:9px; }
.side-bottom { margin-top:auto; border-top:1px solid #D4E8F3; padding-top:10px; color:#7D94A6; font-size:8.5px; line-height:1.5; }
.main-panel { min-width:0; min-height:0; height:100%; display:grid; grid-template-rows:auto auto minmax(245px,1fr) minmax(135px,.56fr) auto; gap:8px; overflow:hidden; }
.kpi-grid { display:grid; grid-template-columns:repeat(5,minmax(0,1fr)); gap:8px; width:100%; min-width:0; }
.kpi-card { border:1px solid #DDECF5; border-top:3px solid var(--accent,#82CFF0); border-radius:12px; background:rgba(255,255,255,.97); min-height:79px; box-shadow:0 3px 10px rgba(92,157,191,.055); padding:8px 9px 7px; overflow:hidden; }
.kpi-title { font-size:clamp(8px,.67vw,10px); font-weight:850; color:#6C8498; line-height:1.25; margin-bottom:5px; min-height:23px; white-space:normal; overflow-wrap:anywhere; }
.kpi-value { font-size:clamp(16px,1.32vw,23px); font-weight:900; color:var(--accent,#82CFF0); line-height:1.05; letter-spacing:-.5px; }
.kpi-sub { font-size:clamp(8px,.60vw,9px); color:#73889A; line-height:1.3; margin-top:5px; white-space:normal; }
.compact-card { border:1px solid #DDECF5; border-radius:12px; background:rgba(255,255,255,.96); box-shadow:0 3px 9px rgba(92,157,191,.045); min-width:0; }
.filter-shell { padding:6px 9px 5px; }
.filter-grid { display:grid; grid-template-columns:.72fr .78fr .88fr 2.15fr auto; gap:7px; align-items:end; }
.filter-label { display:block; font-size:8.5px; font-weight:850; color:#6B8397; margin:0 0 3px 2px; white-space:nowrap; }
.filter-grid .Select-control, .filter-grid .Select__control, .filter-grid .dash-dropdown { min-height:29px!important; height:29px!important; border-radius:7px!important; font-size:10px!important; border-color:#D9E8F2!important; background:#FBFDFF!important; }
.filter-grid .Select-placeholder, .filter-grid .Select-value-label, .filter-grid .Select-input { line-height:27px!important; font-size:10px!important; }
.filter-grid .Select__value-container { padding-top:0!important; padding-bottom:0!important; }
.filter-grid .Select__indicators { height:28px!important; }
.filter-grid .Select-menu-outer, .filter-grid .Select__menu { font-size:10px!important; z-index:1000!important; }
.reset-btn { height:29px; padding:0 9px!important; border-radius:7px!important; font-size:9px!important; white-space:nowrap; border-color:#CFE2EE!important; color:#65849A!important; }
.filter-note { font-size:8px; color:#8BA0B1; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; margin:4px 2px 0; }
.charts-grid { display:grid; grid-template-columns:.9fr 1.15fr; gap:8px; min-height:0; overflow:hidden; }
.chart-card { min-width:0; min-height:0; border:1px solid #DDECF5; border-radius:13px; background:rgba(255,255,255,.98); padding:8px 9px 4px; box-shadow:0 3px 10px rgba(92,157,191,.045); overflow:hidden; display:flex; flex-direction:column; }
.chart-title { flex:0 0 auto; font-size:11.5px; font-weight:900; color:#466781; margin:0 0 2px 2px; }
.chart-sub { flex:0 0 auto; font-size:8.5px; color:#879BAD; margin:0 0 2px 2px; line-height:1.35; }
.guideline-note { flex:0 0 auto; margin:3px 2px 1px; padding:4px 6px; border-radius:7px; background:#F3FAFF; color:#6F8497; font-size:8px; line-height:1.35; max-height:42px; overflow:auto; }
.chart-card .dash-graph { min-height:0!important; flex:1 1 auto; }
.chart-card .js-plotly-plot, .chart-card .plot-container, .chart-card .svg-container { max-height:100%!important; }
.bottom-grid { display:grid; grid-template-columns:1.42fr .86fr; gap:8px; min-height:0; overflow:hidden; }
.table-card { padding:7px 8px 4px; overflow:hidden; }
.bottom-title { font-size:10.5px; font-weight:900; color:#466781; margin:0 0 4px 2px; }
.table-note { font-size:8px; line-height:1.35; color:#879BAD; margin:3px 2px 0; }
.qa-card { padding:8px 10px; overflow:hidden; }
.qa-label { font-size:8.5px; font-weight:850; color:#7B90A2; }
.qa-main { font-size:11px; font-weight:900; color:#526F86; margin:2px 0; }
.qa-text { font-size:8.5px; color:#6C8295; line-height:1.38; }
.qa-alert { border-left:3px solid var(--coral); padding-left:7px; margin:4px 0; }
.source-foot { min-width:0; font-size:7.7px; color:#8BA0B1; line-height:1.3; padding:0 2px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
.dash-table-container .dash-spreadsheet-container .dash-spreadsheet-inner td, .dash-table-container .dash-spreadsheet-container .dash-spreadsheet-inner th { font-size:9px!important; padding:4px 6px!important; }
@media (max-width:1190px) { .dashboard-layout { height:auto; min-height:100vh; max-height:none; overflow:visible; grid-template-columns:145px minmax(0,1fr); } .main-panel { height:auto; overflow:visible; grid-template-rows:auto auto auto auto auto; } .charts-grid { min-height:310px; } .bottom-grid { min-height:150px; } .kpi-title { font-size:9px; } }
@media (max-width:900px) { .dashboard-layout { display:block; padding:8px; } .sidebar { height:auto; margin-bottom:8px; flex-direction:row; align-items:center; gap:10px; padding:8px; } .sidebar-image { width:105px; margin:0; } .side-eyebrow,.side-sub,.side-divider,.side-nav,.side-bottom { display:none; } .side-title { font-size:16px; margin:0; } .main-panel { height:auto; overflow:visible; } .kpi-grid { grid-template-columns:repeat(3,minmax(0,1fr)); } .filter-grid { grid-template-columns:repeat(2,minmax(0,1fr)); } .charts-grid,.bottom-grid { grid-template-columns:1fr; } .chart-card .dash-graph { min-height:270px!important; } .filter-note { white-space:normal; } .source-foot { white-space:normal; } }
@media (max-width:560px) { .kpi-grid { grid-template-columns:repeat(2,minmax(0,1fr)); } .filter-grid { grid-template-columns:1fr 1fr; } .sidebar-image { display:none; } }
</style></head><body>{%app_entry%}<footer>{%config%}{%scripts%}{%renderer%}</footer></body></html>'''

CARD_STYLE = {"border": f"1px solid {PANEL_BORDER}", "borderRadius": "12px", "boxShadow": "0 3px 9px rgba(92,157,191,.045)", "background": "#FFFFFF"}

def kpi_card(title, value, subtitle, accent=SKY):
    return html.Div([html.Div(title, className="kpi-title"), html.Div(value, className="kpi-value"), html.Div(subtitle, className="kpi-sub")], className="kpi-card", style={"--accent": accent})

month_options = [{"label": "전체 기간", "value": "전체"}] + [{"label": m.strftime("%Y년 %m월"), "value": m.strftime("%Y-%m")} for m in months]
supplier_options = [{"label": "전체 공급사", "value": "전체"}] + [{"label": s, "value": s} for s in suppliers]
feed_options = [{"label": "전체 사료", "value": "전체"}] + [{"label": f, "value": f} for f in feed_types]
toxin_options = [
    {"label": "전체 권고대상 독소 · 공개 측정값", "value": "ALL"},
    {"label": RECOMMENDED_TOXINS["DON"]["option"], "value": "DON"},
    {"label": RECOMMENDED_TOXINS["ZEN"]["option"], "value": "ZEN"},
    {"label": RECOMMENDED_TOXINS["FUM"]["option"], "value": "FUM"},
    {"label": RECOMMENDED_TOXINS["T2HT2"]["option"], "value": "T2HT2"},
]

app.layout = dbc.Container(fluid=True, children=[
    html.Div(className="dashboard-layout", children=[
        html.Aside(className="sidebar", children=[
            html.Img(src=SVG_URI, className="sidebar-image", alt="사료 품질 모니터링 일러스트"),
            html.Div("QUALITY ASSURANCE", className="side-eyebrow"),
            html.H1("양돈 사료 곰팡이독소 품질 모니터링", className="side-title"),
            html.P("QA/QC · FEED SAFETY\n원료 · 보관 · 검사 관리", className="side-sub"),
            html.Div(className="side-divider"),
            html.Div([
                html.Div("공개 모니터링", style={"fontWeight":"850","color":"#6189A3","marginBottom":"3px"}),
                html.Div("2026년 1차 양돈용 배합사료", style={"marginBottom":"4px"}),
                html.A("사료 기준·규격 별표 16 ↗", href=RULE_URL, target="_blank", style={"color":"#548EAE","textDecoration":"none","fontWeight":"800"}),
            ], className="side-bottom"),
        ]),
        html.Main(className="main-panel", children=[
            # KPI 5개를 메인 화면 최상단에 배치
            html.Div(className="kpi-grid", children=[
                html.Div(id="kpi-legal", children=kpi_card("01 · 곰팡이독소 기준 적합률", "100%", "아플라·오크라 각각 17건 불검출", GREEN)),
                html.Div(id="kpi-recommendation", children=kpi_card("02 · DON·ZEN 권고기준 초과율", "각 5.9%", "보미톡신 1/17 · 제랄레논 1/17", CORAL)),
                html.Div(id="kpi-inspection", children=kpi_card("03 · 곰팡이독소 검사계획 이행률", "계산 중", "완료 검사 / 계획 검사", SKY_DARK)),
                html.Div(id="kpi-storage", children=kpi_card("04 · 보관환경 관리기준 이탈률", "계산 중", "환경 이탈 / 환경 점검", AMBER)),
                html.Div(id="kpi-sop", children=kpi_card("05 · SOP 준수율", "계산 중", "검사 표준절차 준수", LAVENDER)),
            ]),
            html.Div(className="compact-card filter-shell", children=[
                html.Div(className="filter-grid", children=[
                    html.Div([html.Label("운영 KPI 기간", className="filter-label"), dcc.Dropdown(id="month-filter", options=month_options, value="전체", clearable=False, searchable=False, className="dash-dropdown")]),
                    html.Div([html.Label("공급사", className="filter-label"), dcc.Dropdown(id="supplier-filter", options=supplier_options, value="전체", clearable=False, searchable=False, className="dash-dropdown")]),
                    html.Div([html.Label("사료 종류", className="filter-label"), dcc.Dropdown(id="feed-filter", options=feed_options, value="전체", clearable=False, searchable=False, className="dash-dropdown")]),
                    html.Div([html.Label("권고기준 대상 독소 · 공개 측정값", className="filter-label"), dcc.Dropdown(id="toxin-filter", options=toxin_options, value="ALL", clearable=False, searchable=False, className="dash-dropdown")]),
                    dbc.Button("초기화", id="reset-filters", color="light", outline=True, size="sm", className="reset-btn"),
                ]),
                html.Div("독소 선택에 따라 공개 측정 그래프와 결과표가 바뀝니다. 운영 KPI 필터는 월·공급사·사료 종류에 적용됩니다.", className="filter-note"),
            ]),
            html.Div(className="charts-grid", children=[
                html.Div(className="chart-card", children=[
                    html.Div("독소별 검출 및 기준 초과 건수", className="chart-title"),
                    html.Div("2026년 공개 모니터링 분석대상 4종 · 검체 17건", className="chart-sub"),
                    dcc.Graph(id="public-toxin-chart", figure=fig_public, config={"displayModeBar":False,"responsive":True}, style={"width":"100%","height":"100%"}),
                ]),
                html.Div(className="chart-card", children=[
                    html.Div("권고대상 독소 · 개별 측정값", className="chart-title"),
                    html.Div("하늘빛 = 기준 이내 · 노을빛 = 보도자료상 초과 · 빨간 점선 = 100% 기준선", className="chart-sub"),
                    dcc.Graph(id="public-ratio-chart", figure=make_ratio_fig("ALL"), config={"displayModeBar":False,"responsive":True}, style={"width":"100%","flex":"1 1 auto","minHeight":"0"}),
                    html.Div(id="toxin-guideline-note", className="guideline-note", children="별표 16의 관리 권고 대상은 보미톡신, 제랄레논, 푸모니신, T-2/HT-2입니다. 이번 공개자료에서 측정값이 확인되는 권고 독소는 보미톡신과 제랄레논입니다."),
                ]),
            ]),
            html.Div(className="bottom-grid", children=[
                html.Div(className="compact-card table-card", children=[
                    html.Div("선택한 독소의 공개 측정값", className="bottom-title"),
                    dash_table.DataTable(
                        id="public-results-table", data=[], columns=[
                            {"name":"권고대상 독소", "id":"독소"}, {"name":"농가", "id":"농장"},
                            {"name":"공개 측정값 (ppb)", "id":"측정값"}, {"name":"보도자료 기준 (ppb)", "id":"기사기준"},
                            {"name":"기준 대비", "id":"기준대비"}, {"name":"보도자료 판정", "id":"판정"},
                        ],
                        page_action="none", sort_action="native", style_table={"overflowX":"auto"},
                        style_header={"backgroundColor":"#EAF7FF","fontWeight":"850","color":NAVY,"border":"0","fontSize":"9px","padding":"4px 6px"},
                        style_cell={"fontSize":"9px","padding":"4px 6px","whiteSpace":"nowrap","height":"24px","border":"0","borderBottom":"1px solid #EDF5FA","textAlign":"left","color":"#587084"},
                        style_data_conditional=[{"if":{"filter_query":"{판정} contains '초과'"},"backgroundColor":"#FFF0E9","color":"#B95435","fontWeight":"800"}],
                    ),
                    html.Div(id="public-table-note", className="table-note", children="공개자료에서 확인된 개별 측정값만 표시합니다."),
                ]),
                html.Div(className="compact-card qa-card", children=[
                    html.Div("QA 확인 포인트", className="bottom-title"),
                    html.Div([html.Div("보미톡신(DON) · 권고기준 초과", className="qa-label"), html.Div("1,088.3 / 900 ppb", className="qa-main"), html.Div("F업체 농장 10 · 보도자료 기준 대비 초과", className="qa-text")], className="qa-alert"),
                    html.Div([html.Div("제랄레논(ZEN) · 보도자료상 초과", className="qa-label"), html.Div("134.2 / 100 ppb", className="qa-main"), html.Div("G업체 농장 12 · 급여 단계 미공개", className="qa-text")], className="qa-alert"),
                    html.Div("복합 검출: G·I 농장 2곳. ZEN은 별표 16상 돼지 사료 급여단계별 권고수준이 달라 개별 제품 유형 확인이 필요합니다.", className="qa-text", style={"marginTop":"5px"}),
                ]),
            ]),
            html.Div([
                "운영 KPI 수치는 예시 운영기록으로 계산되며 실제 기업 실적이 아닙니다. 공개 사례 값은 한돈연구소 2026년 모니터링 결과입니다. ",
                html.A("공개 사례 원문", href=CASE_URL, target="_blank", style={"color":"#6097B7","textDecoration":"none","fontWeight":"800"}),
                " · ", html.A("사료 기준·규격 별표 16", href=RULE_URL, target="_blank", style={"color":"#6097B7","textDecoration":"none","fontWeight":"800"}),
            ], className="source-foot"),
        ]),
    ])
])

# 6) 콜백: 운영 KPI와 공개 측정값 필터
@app.callback(
    Output("kpi-inspection", "children"), Output("kpi-storage", "children"), Output("kpi-sop", "children"),
    Input("month-filter", "value"), Input("supplier-filter", "value"), Input("feed-filter", "value")
)
def update_operational_kpis(month_value, supplier_value, feed_value):
    dff = OPS_DF.copy()
    if month_value and month_value != "전체":
        dff = dff[dff["월"] == month_value]
    if supplier_value and supplier_value != "전체":
        dff = dff[dff["공급사"] == supplier_value]
    if feed_value and feed_value != "전체":
        dff = dff[dff["사료종류"] == feed_value]
    if dff.empty:
        return (
            kpi_card("03 · 곰팡이독소 검사계획 이행률", "데이터 없음", "필터 조건을 확인하세요", SKY_DARK),
            kpi_card("04 · 보관환경 관리기준 이탈률", "데이터 없음", "필터 조건을 확인하세요", AMBER),
            kpi_card("05 · SOP 준수율", "데이터 없음", "필터 조건을 확인하세요", LAVENDER),
        )
    n_lots = len(dff)
    inspection_done = int(dff["검사완료"].sum())
    inspection_rate = 100 * float(dff["검사완료"].mean())
    environment_checks = int(dff["환경점검항목수"].sum())
    environment_deviations = int(dff["환경이탈건수"].sum())
    environment_deviation_rate = 100 * environment_deviations / max(environment_checks, 1)
    sop_pass_count = int(dff["SOP준수"].sum())
    sop_rate = 100 * float(dff["SOP준수"].mean())
    return (
        kpi_card("03 · 곰팡이독소 검사계획 이행률", f"{inspection_rate:.1f}%", f"검사 완료 {inspection_done}/{n_lots} 로트", SKY_DARK),
        kpi_card("04 · 보관환경 관리기준 이탈률", f"{environment_deviation_rate:.1f}%", f"이탈 {environment_deviations}/{environment_checks}개 점검", AMBER if environment_deviation_rate >= 8 else GREEN),
        kpi_card("05 · SOP 준수율", f"{sop_rate:.1f}%", f"준수 {sop_pass_count}/{n_lots} 로트", LAVENDER if sop_rate >= 95 else CORAL),
    )


def public_table_rows(toxin_choice="ALL"):
    dff = PUBLIC_KNOWN_RESULTS.copy()
    if toxin_choice in ["DON", "ZEN"]:
        dff = dff[dff["독소코드"] == toxin_choice]
    elif toxin_choice in ["FUM", "T2HT2"]:
        dff = dff.iloc[0:0]
    elif toxin_choice != "ALL":
        dff = dff.iloc[0:0]
    dff = dff.sort_values("기준대비_pct", ascending=False)
    rows = []
    for _, r in dff.iterrows():
        rows.append({"독소": r["독소"], "농장": r["농장"], "측정값": f"{r['농도_ppb']:,.2f}",
                     "기사기준": f"{r['기사기준_ppb']:,.0f}", "기준대비": f"{r['기준대비_pct']:.1f}%", "판정": r["결과"]})
    return rows


def guideline_note(toxin_choice="ALL"):
    if toxin_choice == "ALL":
        return "별표 16 관리 권고 대상 4종: 보미톡신(DON) 900 ppb(돼지용 배합사료), 제랄레논(ZEN) 100/250 ppb(돼지 급여단계별), 푸모니신(B1+B2) 5,000 ppb(돼지용 배합사료), T-2/HT-2 250 ppb(배합사료). 이번 공개사례의 개별 측정값은 DON·ZEN만 확인됩니다."
    item = RECOMMENDED_TOXINS.get(toxin_choice)
    if item is None:
        return "권고수준은 사료 종류에 따라 다를 수 있습니다."
    return f"별표 16 관리 권고수준: {item['standard']}. {item['public_note']}"

@app.callback(
    Output("public-ratio-chart", "figure"), Output("public-results-table", "data"),
    Output("toxin-guideline-note", "children"), Output("public-table-note", "children"),
    Input("toxin-filter", "value")
)
def update_public_measurements(toxin_value):
    choice = toxin_value or "ALL"
    rows = public_table_rows(choice)
    if choice in ["FUM", "T2HT2"]:
        table_note = "이번 공개 모니터링의 분석 대상에 포함되지 않아 공개 측정값이 없습니다. 측정값을 임의로 생성하지 않았습니다."
    elif not rows:
        table_note = "선택한 항목에 해당하는 공개 개별 측정값이 없습니다."
    else:
        table_note = "공개자료의 개별 측정값만 표시합니다. ZEN은 보도자료 적용 기준과 별표 16의 급여단계별 권고수준을 구분해 해석해야 합니다."
    return make_ratio_fig(choice), rows, guideline_note(choice), table_note

@app.callback(
    Output("month-filter", "value"), Output("supplier-filter", "value"), Output("feed-filter", "value"), Output("toxin-filter", "value"),
    Input("reset-filters", "n_clicks"), prevent_initial_call=True
)
def reset_filters(_):
    return "전체", "전체", "전체", "ALL"
# 7) Run dashboard; in Colab, create a clickable link for a new browser tab

def find_free_port(start=8050, end=8099):
    for p in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("0.0.0.0", p))
                return p
            except OSError:
                continue
    raise RuntimeError("사용 가능한 포트를 찾지 못했습니다. 런타임을 재시작한 뒤 다시 실행하세요.")

PORT = find_free_port()
def run_dashboard_server():
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)

try:
    from google.colab import output as colab_output
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    server_thread = threading.Thread(target=run_dashboard_server, daemon=True)
    server_thread.start()
    server_ready = False
    for _ in range(40):
        try:
            with socket.create_connection(("127.0.0.1", PORT), timeout=0.5):
                server_ready = True
                break
        except OSError:
            time.sleep(0.5)
    if not server_ready:
        raise RuntimeError("대시보드 서버가 시작되지 않았습니다. 셀을 다시 실행하거나 런타임을 재시작하세요.")
    from IPython.display import display, HTML
    from html import escape
    print("대시보드가 실행되었습니다. 아래 링크를 누르면 새 브라우저 탭에서 열립니다.")
    try:
        proxy_url = colab_output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
        if not isinstance(proxy_url, str) or not proxy_url.startswith(("http://", "https://")):
            raise RuntimeError("Colab 프록시 URL을 가져오지 못했습니다.")
        safe_url = escape(proxy_url, quote=True)
        display(HTML(
            '<div style="font-family:Arial,sans-serif;padding:13px 16px;border:1px solid #d9e3ed;'
            'border-radius:12px;background:#f8fbff;max-width:650px">'
            '<div style="font-size:12px;color:#52677d;margin-bottom:8px">양돈 사료 곰팡이독소 QA/QC</div>'
            f'<a href="{safe_url}" target="_blank" rel="noopener noreferrer" style="display:inline-block;'
            'background:linear-gradient(110deg,#82CFF0,#C9C7F5);color:white;text-decoration:none;'
            'font-size:14px;font-weight:700;padding:11px 17px;border-radius:9px">대시보드 새 탭 열기 ↗</a>'
            '<div style="font-size:10px;color:#718198;margin-top:8px">현재 Colab 런타임이 실행 중일 때 열 수 있습니다.</div></div>'
        ))
        print("대시보드 링크:", proxy_url)
    except Exception as exc:
        print("클릭 링크 생성에 실패했습니다. 아래 포트 링크를 새 탭에서 열어 주세요.")
        try:
            proxy_url = colab_output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
            print(proxy_url)
        except Exception:
            try:
                colab_output.serve_kernel_port_as_window(PORT, anchor_text="대시보드 새 탭 열기", skip_warning=True)
            except Exception:
                print(f"대시보드 서버는 포트 {PORT}에서 실행 중입니다. 오류: {exc}")
else:
    print(f"대시보드 주소: http://127.0.0.1:{PORT}")
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)


<IPython.core.display.Javascript object>

대시보드가 실행되었습니다. 아래 링크를 누르면 새 브라우저 탭에서 열립니다.


대시보드 링크: https://8053-m-s-ase1a6-3ckeuoaje84ec-a.asia-east1.prod.r.colab.dev
